# Построение рекомендательной системы

В этом блокноте мы построим модель рекомендательной системы для кластеров отелей.

In [1]:
# Импорт необходимых модулей
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

## Шаг 1: Чтение и обработка обучающих данных

In [2]:
# Чтение обучающих данных и вычисление агрегированных значений
train = pd.read_csv('data/train.csv',
                    dtype={'is_booking':bool,'srch_destination_id':np.int32, 'hotel_cluster':np.int32},
                    usecols=['srch_destination_id','is_booking','hotel_cluster'],
                    chunksize=100000)

aggs = []

for chunk in tqdm(train, desc='Processing train chunks'):
    agg = chunk.groupby(['srch_destination_id', 'hotel_cluster'])['is_booking'].agg(['sum','count'])
    agg.reset_index(inplace=True)
    aggs.append(agg)
aggs = pd.concat(aggs, axis=0)

Processing train chunks: 0it [00:00, ?it/s]

## Шаг 2: Вычисление релевантности кластеров отелей

In [3]:
CLICK_WEIGHT = 0.05

# Группируем снова для суммирования по всем чанкам
agg = aggs.groupby(['srch_destination_id','hotel_cluster']).sum().reset_index()

# Вычисляем количество кликов
agg['clicks'] = agg['count'] - agg['sum']

# Переименовываем столбец sum в bookings
agg = agg.rename(columns={'sum':'bookings'})

# Вычисляем релевантность
agg['relevance'] = agg['bookings'] + CLICK_WEIGHT * agg['clicks']

### Функция для определения наиболее популярных отелей для конкретного srch_destination_id

In [4]:
def most_popular(group, n_max=5):
    relevance = group['relevance'].values
    hotel_cluster = group['hotel_cluster'].values
    most_popular_clusters = hotel_cluster[np.argsort(-relevance)][:n_max]
    return list(most_popular_clusters)

### Получаем наиболее популярные кластеры отелей для всех направлений

In [5]:
most_pop = agg.groupby('srch_destination_id').apply(most_popular).reset_index()
most_pop.columns = ['srch_destination_id', 'hotel_cluster']

C:\Users\Gleb\AppData\Local\Temp\ipykernel_6532\899261877.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  most_pop = agg.groupby('srch_destination_id').apply(most_popular).reset_index()


## Шаг 3: Формируем предсказания для тестовой выборки

In [6]:
# Чтение тестовых данных
test = pd.read_csv('data/test.csv')
test = test[['srch_destination_id']]  # Используем только необходимый столбец

# Объединяем тестовые данные с наиболее популярными кластерами
test = test.merge(most_pop, how='left', on='srch_destination_id')

# Заполним NaN наиболее популярными кластерами в целом
most_pop_all = agg.groupby('hotel_cluster')['relevance'].sum().nlargest(5).index.tolist()
test['hotel_cluster'] = test['hotel_cluster'].apply(lambda x: x if isinstance(x, list) else most_pop_all)

## Шаг 4: Формирование финального списка предсказаний

In [9]:
# Получаем список предсказаний
predictions = test['hotel_cluster'].tolist()
# Преобразуем np int32 в int для сериализации
predictions = [list(map(int, x)) for x in predictions]

# Проверяем формат предсказаний
print(predictions[:5])  # Выводим первые 5 предсказаний для проверки

[[89, 53], [82, 28, 36, 46, 15], [97, 58, 25, 10, 64], [63, 36, 44, 62, 57], [98, 70, 41, 19, 56]]


## Шаг 5: Сохранение предсказаний (опционально)

In [11]:
# Сохранение предсказаний в формате JSON
import json

with open('predictions.json', 'w') as f:
    json.dump(predictions, f)